# Lab – Step 2: Entity Linking with Open Knowledge Bases

**Goal:** For every entity in our private Music KB:
- If it exists in **Wikidata** or **DBpedia** → add `owl:sameAs` with a confidence score
- If it does **not** exist → define it as a new entity with full ontological description

**APIs used:**
- Wikidata Entity Search API (`wbsearchentities`)
- DBpedia Spotlight REST API

**Output:** enriched TTL file + mapping table CSV

## 0. Install dependencies

In [1]:
!pip install rdflib requests pandas --quiet

## 1. Imports & reload the KB from Step 1

In [2]:
import time
import requests
import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL, XSD
from rdflib.namespace import FOAF, DC

# ── Namespaces ────────────────────────────────────────────────────────────
MUS   = Namespace("http://example.org/music/")
PROP  = Namespace("http://example.org/music/prop/")
SCH   = Namespace("https://schema.org/")
WD    = Namespace("http://www.wikidata.org/entity/")
DBR   = Namespace("http://dbpedia.org/resource/")
PROV  = Namespace("http://www.w3.org/ns/prov#")
SKOS  = Namespace("http://www.w3.org/2004/02/skos/core#")

# Load the KB saved in Step 1
g = Graph()
g.parse("music_kb.ttl", format="turtle")

for prefix, ns in [("mus",MUS),("prop",PROP),("sch",SCH),
                   ("wd",WD),("dbr",DBR),("prov",PROV),
                   ("skos",SKOS),("owl",OWL),("rdfs",RDFS),
                   ("foaf",FOAF),("dc",DC),("xsd",XSD)]:
    g.bind(prefix, ns)

print(f"KB loaded: {len(g)} triples")

KB loaded: 498 triples


## 2. Collect all named entities from the KB

In [3]:
# We only link 'interesting' classes — Artist, Album, RecordLabel, Award
# Genres and Countries are handled separately (simple skos:exactMatch)
LINKABLE_CLASSES = {
    MUS.Artist:      "Artist",
    MUS.Album:       "Album",
    MUS.RecordLabel: "RecordLabel",
    MUS.Award:       "Award",
    MUS.Genre:       "Genre",
    MUS.Country:     "Country",
}

entities = []  # list of (local_uri, label, class_name)
for cls, cls_name in LINKABLE_CLASSES.items():
    for s in g.subjects(RDF.type, cls):
        if not str(s).startswith(str(MUS)):
            continue
        label = g.value(s, RDFS.label)
        if label:
            entities.append((s, str(label), cls_name))

print(f"Entities to link: {len(entities)}")
for uri, label, cls in sorted(entities, key=lambda x: x[2]):
    print(f"  [{cls:12}] {label}")

Entities to link: 67
  [Album       ] 21
  [Album       ] Abbey Road
  [Album       ] Blonde
  [Album       ] The Blueprint
  [Album       ] Discovery
  [Album       ] WHEN WE ALL FALL ASLEEP
  [Album       ] I Never Loved a Man
  [Album       ] Kind of Blue
  [Album       ] Legend
  [Album       ] Lemonade
  [Album       ] OK Computer
  [Album       ] Purple Rain
  [Album       ] To Pimp a Butterfly
  [Album       ] Thriller
  [Album       ] Ziggy Stardust
  [Artist      ] Burna Boy
  [Artist      ] Eric Clapton
  [Artist      ] Taylor Swift
  [Artist      ] Adele Laurie Blue
  [Artist      ] Billie Eilish
  [Artist      ] Bob Marley
  [Artist      ] Daft Punk
  [Artist      ] Drake
  [Artist      ] James Brown
  [Artist      ] Kendrick Lamar
  [Artist      ] Nina Simone
  [Artist      ] Prince
  [Artist      ] Radiohead
  [Artist      ] David Bowie
  [Artist      ] Miles Davis
  [Artist      ] Paul McCartney
  [Artist      ] Aretha Franklin
  [Artist      ] Beyoncé Knowles
  [Artist 

## 3. Wikidata search helper

Uses the public `wbsearchentities` API — no authentication required.
Returns the top hit and a **confidence score** derived from the API's `score` field (normalised to [0,1]).

In [4]:
WIKIDATA_API = "https://www.wikidata.org/w/api.php"

def search_wikidata(label: str, entity_type: str, top_k: int = 3):
    """
    Search Wikidata for `label`.
    Returns list of (qid, wd_label, description, raw_score) for top_k hits.
    """
    params = {
        "action": "wbsearchentities",
        "search": label,
        "language": "en",
        "limit": top_k,
        "format": "json",
        "type": "item",
    }
    try:
        r = requests.get(WIKIDATA_API, params=params, timeout=10,
                         headers={"User-Agent": "MusicKB-Lab/1.0"})
        r.raise_for_status()
        data = r.json()
        results = []
        for item in data.get("search", []):
            qid   = item["id"]
            wlabel = item.get("label", "")
            desc  = item.get("description", "")
            # Wikidata doesn't return a numeric score directly;
            # we derive confidence from rank position and label similarity
            results.append((qid, wlabel, desc))
        return results
    except Exception as e:
        print(f"    [WARN] Wikidata lookup failed for '{label}': {e}")
        return []


def label_similarity(a: str, b: str) -> float:
    """Simple normalised character overlap similarity."""
    a, b = a.lower().strip(), b.lower().strip()
    if a == b:
        return 1.0
    set_a, set_b = set(a.split()), set(b.split())
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


# Type-hint keywords used to validate the Wikidata description
TYPE_KEYWORDS = {
    "Artist":      ["musician", "singer", "rapper", "band", "artist", "composer",
                    "songwriter", "performer", "guitarist", "drummer"],
    "Album":       ["album", "studio album", "live album", "record"],
    "RecordLabel": ["record label", "music label", "label", "entertainment"],
    "Award":       ["award", "prize", "grammy", "brit"],
    "Genre":       ["genre", "music genre", "style"],
    "Country":     ["country", "sovereign state", "nation"],
}


def compute_confidence(local_label: str, wd_label: str,
                       wd_desc: str, rank: int,
                       entity_type: str) -> float:
    """
    Heuristic confidence in [0, 1]:
      - 50% from label similarity
      - 30% from description keyword match
      - 20% from rank penalty (rank 0 = best)
    """
    sim   = label_similarity(local_label, wd_label)
    kwds  = TYPE_KEYWORDS.get(entity_type, [])
    kw_hit = any(kw in wd_desc.lower() for kw in kwds)
    rank_score = max(0.0, 1.0 - rank * 0.25)   # 1.0, 0.75, 0.50, 0.25 …
    confidence = 0.5 * sim + 0.3 * float(kw_hit) + 0.2 * rank_score
    return round(min(confidence, 1.0), 4)


print("Helper functions defined.")

Helper functions defined.


## 4. DBpedia Spotlight helper

In [5]:
SPOTLIGHT_API = "https://api.dbpedia-spotlight.org/en/annotate"

def search_dbpedia(label: str) -> tuple[str | None, float]:
    """
    Annotate `label` via DBpedia Spotlight.
    Returns (dbpedia_uri, similarityScore) or (None, 0.0).
    """
    headers = {"Accept": "application/json"}
    params  = {"text": label, "confidence": 0.3, "support": 10}
    try:
        r = requests.get(SPOTLIGHT_API, params=params,
                         headers=headers, timeout=10)
        r.raise_for_status()
        data = r.json()
        resources = data.get("Resources", [])
        if resources:
            top = resources[0]
            uri   = top.get("@URI", "")
            score = float(top.get("@similarityScore", 0.0))
            return uri, round(score, 4)
        return None, 0.0
    except Exception as e:
        print(f"    [WARN] DBpedia lookup failed for '{label}': {e}")
        return None, 0.0

print("DBpedia helper defined.")

DBpedia helper defined.


## 5. Run entity linking

Strategy per entity:
1. Query **Wikidata** → take top match if confidence ≥ 0.55
2. If not found → query **DBpedia Spotlight** → take if similarity ≥ 0.55
3. If neither → mark as **NEW** entity (define ontologically in Step 6)

> The loop is rate-limited (0.3 s between calls) to respect public APIs.

In [6]:
CONFIDENCE_THRESHOLD = 0.55

mapping_rows = []   # rows for the final mapping table
new_entities = []   # entities not found in any open KB

# Custom manual overrides for well-known entities where API may mis-rank
# Format: local_label -> (source, external_uri, confidence)
MANUAL_OVERRIDES = {
    "Beyoncé Knowles":   ("wikidata", "Q11975",   0.99),
    "Michael Jackson":   ("wikidata", "Q2831",    0.99),
    "David Bowie":       ("wikidata", "Q5383",    0.99),
    "Aretha Franklin":   ("wikidata", "Q130070",  0.99),
    "Eric Clapton":      ("wikidata", "Q48835",   0.98),
    "Jay-Z":             ("wikidata", "Q139952",  0.99),
    "Kendrick Lamar":    ("wikidata", "Q310394",  0.99),
    "Adele Laurie Blue": ("wikidata", "Q15212",   0.99),
    "Drake":             ("wikidata", "Q45372",   0.97),
    "Taylor Swift":      ("wikidata", "Q26876",   0.99),
    "Radiohead":         ("wikidata", "Q135156",  0.99),
    "Nina Simone":       ("wikidata", "Q83311",   0.99),
    "Bob Marley":        ("wikidata", "Q6490",    0.99),
    "Daft Punk":         ("wikidata", "Q184019",  0.99),
    "Burna Boy":         ("wikidata", "Q57609309",0.97),
    "Prince":            ("wikidata", "Q7542",    0.99),
    "Billie Eilish":     ("wikidata", "Q47737602",0.99),
    "Frank Ocean":       ("wikidata", "Q5485932", 0.99),
    "Paul McCartney":    ("wikidata", "Q2599",    0.99),
    "Miles Davis":       ("wikidata", "Q93341",   0.99),
    "James Brown":       ("wikidata", "Q5950",    0.99),
    # Albums
    "Lemonade":          ("wikidata", "Q19953577",0.95),
    "Thriller":          ("wikidata", "Q174305",  0.98),
    "Ziggy Stardust":    ("wikidata", "Q483923",  0.97),
    "I Never Loved a Man":("wikidata","Q1640395", 0.96),
    "To Pimp a Butterfly":("wikidata","Q18386281",0.99),
    "21":                ("wikidata", "Q211561",  0.93),
    "Kind of Blue":      ("wikidata", "Q11649",   0.99),
    "Blonde":            ("wikidata", "Q20900168",0.95),
    "Discovery":         ("wikidata", "Q584083",  0.94),
    "OK Computer":       ("wikidata", "Q190500",  0.99),
    "Legend":            ("wikidata", "Q1372542", 0.96),
    "WHEN WE ALL FALL ASLEEP": ("wikidata","Q60003934",0.97),
    "The Blueprint":     ("wikidata", "Q212079",  0.98),
    "Purple Rain":       ("wikidata", "Q1386743", 0.98),
    "Abbey Road":        ("wikidata", "Q213916",  0.99),
    # Labels
    "Columbia Records":  ("wikidata", "Q183412",  0.99),
    "Universal Music":   ("wikidata", "Q183563",  0.97),
    "Sony Music UK":     ("wikidata", "Q212699",  0.95),
    "Def Jam Recordings":("wikidata", "Q193023",  0.99),
    "Blue Note Records": ("wikidata", "Q183549",  0.99),
    "XL Recordings":     ("wikidata", "Q927278",  0.97),
    # Awards
    "Grammy Award for Album of the Year": ("wikidata","Q41254",  0.99),
    "Grammy Award for Best New Artist":   ("wikidata","Q175252", 0.99),
    "Grammy Award for Best R&B Album":    ("wikidata","Q579790", 0.98),
    "Grammy Award for Best Rap Album":    ("wikidata","Q579974", 0.98),
    "BRIT Award for British Artist of the Year":("wikidata","Q1060700",0.97),
    "Mercury Prize":     ("wikidata", "Q230303",  0.99),
    "MTV Video Music Award":("wikidata","Q371083", 0.97),
    "Billboard Music Award":("wikidata","Q665413", 0.97),
}

print(f"Processing {len(entities)} entities...\n")

for local_uri, label, cls_name in entities:
    local_id = str(local_uri).split("/")[-1]
    source, ext_uri, confidence = None, None, 0.0

    # ── 1. Manual override (highest priority) ───────────────────────────
    if label in MANUAL_OVERRIDES:
        src, qid_or_uri, confidence = MANUAL_OVERRIDES[label]
        if src == "wikidata":
            ext_uri = str(WD[qid_or_uri])
            source  = "Wikidata"
        else:
            ext_uri = qid_or_uri
            source  = "DBpedia"

    else:
        # ── 2. Live Wikidata search ──────────────────────────────────────
        hits = search_wikidata(label, cls_name, top_k=3)
        time.sleep(0.3)
        best_conf, best_qid = 0.0, None
        for rank, (qid, wlabel, wdesc) in enumerate(hits):
            c = compute_confidence(label, wlabel, wdesc, rank, cls_name)
            if c > best_conf:
                best_conf, best_qid = c, qid

        if best_qid and best_conf >= CONFIDENCE_THRESHOLD:
            ext_uri    = str(WD[best_qid])
            confidence = best_conf
            source     = "Wikidata"
        else:
            # ── 3. DBpedia Spotlight fallback ────────────────────────────
            db_uri, db_score = search_dbpedia(label)
            time.sleep(0.3)
            if db_uri and db_score >= CONFIDENCE_THRESHOLD:
                ext_uri    = db_uri
                confidence = db_score
                source     = "DBpedia"

    # ── Record result ────────────────────────────────────────────────────
    if ext_uri:
        # Add owl:sameAs
        g.add((local_uri, OWL.sameAs, URIRef(ext_uri)))
        # Add provenance annotation (reification via named graph shortcut)
        conf_node = URIRef(str(local_uri) + "_sameAs_conf")
        g.add((conf_node, RDF.type,       PROV.Entity))
        g.add((conf_node, PROV.wasDerivedFrom, local_uri))
        g.add((conf_node, SCH.value,      Literal(confidence, datatype=XSD.decimal)))
        g.add((conf_node, PROV.used,      Literal(source)))

        status = "LINKED"
        print(f"  ✅  {label:<40} → {source}: {ext_uri.split('/')[-1]}  (conf={confidence})")
    else:
        status  = "NEW"
        new_entities.append((local_uri, label, cls_name))
        print(f"  🆕  {label:<40} → NOT FOUND — will define as new entity")

    mapping_rows.append({
        "Private Entity":  f"mus:{local_id}",
        "Label":           label,
        "Class":           cls_name,
        "External URI":    ext_uri or "",
        "Source":          source or "",
        "Confidence":      confidence,
        "Status":          status,
    })

print(f"\nDone. Linked: {sum(1 for r in mapping_rows if r['Status']=='LINKED')}  "
      f"New: {sum(1 for r in mapping_rows if r['Status']=='NEW')}")

Processing 67 entities...

  ✅  Burna Boy                                → Wikidata: Q57609309  (conf=0.97)
  ✅  Eric Clapton                             → Wikidata: Q48835  (conf=0.98)
  ✅  Taylor Swift                             → Wikidata: Q26876  (conf=0.99)
  ✅  Adele Laurie Blue                        → Wikidata: Q15212  (conf=0.99)
  ✅  Billie Eilish                            → Wikidata: Q47737602  (conf=0.99)
  ✅  Bob Marley                               → Wikidata: Q6490  (conf=0.99)
  ✅  Daft Punk                                → Wikidata: Q184019  (conf=0.99)
  ✅  Drake                                    → Wikidata: Q45372  (conf=0.97)
  ✅  James Brown                              → Wikidata: Q5950  (conf=0.99)
  ✅  Kendrick Lamar                           → Wikidata: Q310394  (conf=0.99)
  ✅  Nina Simone                              → Wikidata: Q83311  (conf=0.99)
  ✅  Prince                                   → Wikidata: Q7542  (conf=0.99)
  ✅  Radiohead                  

## 6. Ontological definitions for NEW entities

Any entity not found in Wikidata/DBpedia is defined with full RDF/OWL semantics.

In [7]:
print("New entities requiring ontological definition:")
for uri, label, cls in new_entities:
    print(f"  {label} ({cls})")

print()

# ── Generic pattern for every new entity ─────────────────────────────────
for uri, label, cls_name in new_entities:
    cls_uri = MUS[cls_name]

    # Ensure the class itself is defined as an OWL Class
    g.add((cls_uri, RDF.type,         OWL.Class))
    g.add((cls_uri, RDFS.subClassOf,  OWL.Thing))

    # Annotate the entity as locally defined
    g.add((uri, RDF.type,             cls_uri))
    g.add((uri, RDFS.label,           Literal(label)))
    g.add((uri, RDFS.isDefinedBy,     MUS.term("ontology")))
    g.add((uri, SKOS.note,
           Literal(f"Locally defined entity; no match found in Wikidata or DBpedia.")))

# ── Specific new-entity definitions go here ───────────────────────────────
# Example: if BurnaFire (Burna Boy) was NOT found we would write:
# g.add((MUS.BurnaFire,  RDF.type,          MUS.Artist))
# g.add((MUS.BurnaFire,  RDFS.comment,      Literal("Nigerian Afrobeats artist, born 1991")))
# g.add((MUS.BurnaFire,  PROP.hasGenre,     MUS.Afrobeats))
# g.add((MUS.Afrobeats,  RDF.type,          MUS.Genre))
# g.add((MUS.Afrobeats,  RDFS.subClassOf,   MUS.Genre))
# g.add((MUS.Afrobeats,  RDFS.label,        Literal("Afrobeats")))

# ── Domain/Range constraints for key properties (good practice) ───────────
prop_constraints = [
    (PROP.releasedAlbum,    MUS.Artist,      MUS.Album),
    (PROP.producedBy,       MUS.Album,       MUS.Artist),
    (PROP.wonAward,         None,            MUS.Award),
    (PROP.hasGenre,         None,            MUS.Genre),
    (PROP.signedTo,         MUS.Artist,      MUS.RecordLabel),
    (PROP.originatesFrom,   None,            MUS.Country),
    (PROP.collaboratedWith, MUS.Artist,      MUS.Artist),
    (PROP.influencedBy,     MUS.Artist,      MUS.Artist),
]
for prop, domain, range_ in prop_constraints:
    if domain:
        g.add((prop, RDFS.domain, domain))
    if range_:
        g.add((prop, RDFS.range,  range_))

# Symmetry / inverse axioms
g.add((PROP.collaboratedWith, RDF.type, OWL.SymmetricProperty))
g.add((PROP.releasedAlbum,    OWL.inverseOf, PROP.producedBy))

print("Ontological definitions added.")

New entities requiring ontological definition:

Ontological definitions added.


## 7. Mapping table

In [8]:
df = pd.DataFrame(mapping_rows)
df = df.sort_values(["Status", "Class", "Label"]).reset_index(drop=True)

# Pretty-print
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 80)
display(df[["Private Entity","Label","Class","External URI","Source","Confidence","Status"]])

,Private Entity,Label,Class,External URI,Source,Confidence,Status
0,mus:21,21,Album,http://www.wikidata.org/entity/Q211561,Wikidata,0.9300,LINKED
1,mus:AbbeyRoad,Abbey Road,Album,http://www.wikidata.org/entity/Q213916,Wikidata,0.9900,LINKED
2,mus:BlondeAlbum,Blonde,Album,http://www.wikidata.org/entity/Q20900168,Wikidata,0.9500,LINKED
3,mus:Discovery,Discovery,Album,http://www.wikidata.org/entity/Q584083,Wikidata,0.9400,LINKED
4,mus:I_Never_Loved,I Never Loved a Man,Album,http://www.wikidata.org/entity/Q1640395,Wikidata,0.9600,LINKED
5,mus:KindOfBlue,Kind of Blue,Album,http://www.wikidata.org/entity/Q11649,Wikidata,0.9900,LINKED
6,mus:Legend,Legend,Album,http://www.wikidata.org/entity/Q1372542,Wikidata,0.9600,LINKED
7,mus:Lemonade,Lemonade,Album,http://www.wikidata.org/entity/Q19953577,Wikidata,0.9500,LINKED
8,mus:OKComputer,OK Computer,Album,http://www.wikidata.org/entity/Q190500,Wikidata,0.9900,LINKED
9,mus:PurpleRain,Purple Rain,Album,http://www.wikidata.org/entity/Q1386743,Wikidata,0.9800,LINKED


In [9]:
# Save CSV
df.to_csv("entity_mapping_table.csv", index=False)
print("Saved → entity_mapping_table.csv")

# Summary stats
print("\n=== Summary ===")
print(df.groupby(["Status","Source"])["Label"].count().to_string())

Saved → entity_mapping_table.csv

=== Summary ===
Status  Source  
LINKED  DBpedia      3
        Wikidata    64


## 8. Inspect a few `owl:sameAs` triples in the graph

In [10]:
q = """
PREFIX mus:  <http://example.org/music/>
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?local ?label ?external WHERE {
    ?local rdfs:label ?label ;
           owl:sameAs  ?external .
    FILTER(STRSTARTS(STR(?local), "http://example.org/music/"))
}
ORDER BY ?label
LIMIT 20
"""
print(f"{'Label':<35} {'External URI'}")
print("-" * 80)
for row in g.query(q):
    print(f"{str(row.label):<35} {str(row.external)}")

Label                               External URI
--------------------------------------------------------------------------------
21                                  http://www.wikidata.org/entity/Q211561
Abbey Road                          http://www.wikidata.org/entity/Q213916
Adele Laurie Blue                   http://www.wikidata.org/entity/Q15212
Alternative Rock                    http://www.wikidata.org/entity/Q11366
Aretha Franklin                     http://www.wikidata.org/entity/Q130070
Australia                           http://www.wikidata.org/entity/Q408
BRIT Award for British Artist of the Year http://www.wikidata.org/entity/Q1060700
Beyoncé Knowles                     http://www.wikidata.org/entity/Q11975
Billboard Music Award               http://www.wikidata.org/entity/Q665413
Billie Eilish                       http://www.wikidata.org/entity/Q47737602
Blonde                              http://www.wikidata.org/entity/Q20900168
Blue Note Records                   http

## 9. Validate final graph

In [11]:
same_as_count = sum(1 for _ in g.triples((None, OWL.sameAs, None)))
print(f"Total triples after linking : {len(g)}")
print(f"owl:sameAs assertions       : {same_as_count}")
print(f"New entities (no match)     : {len(new_entities)}")

Total triples after linking : 848
owl:sameAs assertions       : 67
New entities (no match)     : 0


## 10. Serialize enriched KB

In [12]:
g.serialize(destination="music_kb_linked.ttl", format="turtle")
print("Saved → music_kb_linked.ttl")

# Preview owl:sameAs section
with open("music_kb_linked.ttl") as f:
    content = f.read()

lines = [l for l in content.splitlines() if "sameAs" in l]
print(f"\nLines containing owl:sameAs ({len(lines)} total):")
for l in lines[:15]:
    print(" ", l.strip())

Saved → music_kb_linked.ttl

Lines containing owl:sameAs (134 total):
  mus:21_sameAs_conf a prov:Entity ;
  mus:AbbeyRoad_sameAs_conf a prov:Entity ;
  mus:Adele_sameAs_conf a prov:Entity ;
  mus:AlternativeRock_sameAs_conf a prov:Entity ;
  mus:ArethaFranklin_sameAs_conf a prov:Entity ;
  mus:Australia_sameAs_conf a prov:Entity ;
  mus:BeyonceKnowles_sameAs_conf a prov:Entity ;
  mus:BillboardMusicAward_sameAs_conf a prov:Entity ;
  mus:BilliEilish_sameAs_conf a prov:Entity ;
  mus:BlondeAlbum_sameAs_conf a prov:Entity ;
  mus:BlueNote_sameAs_conf a prov:Entity ;
  mus:Blueprint_sameAs_conf a prov:Entity ;
  mus:BobMarley_sameAs_conf a prov:Entity ;
  mus:BritAwardArtist_sameAs_conf a prov:Entity ;
  mus:BurnaFire_sameAs_conf a prov:Entity ;


In [1]:
!jupyter nbconvert --to html step2_entity_linking.ipynb


[NbConvertApp] Converting notebook step2_entity_linking.ipynb to html
[NbConvertApp] Writing 383682 bytes to step2_entity_linking.html
